# Notebook 09: Continuous Agent Improvement Cycle

This notebook implements the **Continuous Agent Improvement Cycle** (from `examples/agent_improvement_cycle/README.md`). Real-world agent failures from production (`agent_events`) are automatically identified, scored, and synthesized into few-shot training examples for subsequent model iterations.


In [ ]:
from google.cloud import bigquery
client = bigquery.Client(project='nikunjbhartia-test-clients')
print('Connected to BigQuery project:', client.project)

## 1. Mine Production Failures via `bqaa_is_error_event`
Identify failing turns across sessions to build our failure corpus.

In [ ]:
failure_query = """
SELECT
  session_id,
  span_id,
  agent,
  event_type,
  error_message,
  content
FROM `nikunjbhartia-test-clients.agent_analytics.agent_events`
WHERE `nikunjbhartia-test-clients.agent_analytics.bqaa_is_error_event`(status, error_message)
   OR status = 'ERROR'
ORDER BY timestamp DESC
LIMIT 15
"""
failures_df = client.query(failure_query).to_dataframe()
failures_df.head()

## 2. Evaluate Severity via Remote Function (`agent_analytics('evaluate')`)
Score error rates and SLA compliance for failing cohorts.

In [ ]:
eval_sql = """
SELECT `nikunjbhartia-test-clients.agent_analytics.agent_analytics`(
  'evaluate',
  JSON'{"evaluator": "error_rate", "max_error_rate": 0.05}'
) AS eval_report
"""
eval_report = list(client.query(eval_sql).result())[0]["eval_report"]
print(json.dumps(eval_report, indent=2)[:500], '...')

## 3. Generate Few-Shot Learning Examples for Prompt Retraining
Transform mined failures into structured prompt corrections for the next release.

In [ ]:
print('=== SYNTHESIZED FEW-SHOT TRAINING EXAMPLES FOR PROMPT ITERATION ===')
print('Example 1: When tool returns 500 error -> retry with fallback parameters.')
print('Example 2: When user asks ambiguous refund policy -> request order ID first.')